# Organoids: cell metadata

Joins the expression and site-count metadata and fixes the cell-level annotations (induction status, lineage, library and sequencing batch). Lineage sizes: 47,090 excitatory and 23,740 inhibitory cells; 14 induced and 8 uninduced libraries.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from pathlib import Path

In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/organoids/knoblich"
METADATA_PATH = f"{DATA_ROOT}/3utr/organoids/knoblich/metadata"

COUNTS_PATH = f"{BASEPATH}/cellranger"
COUNTS_FILE = f"{COUNTS_PATH}/aggregated_annotated_filtered_normalized.h5ad"

UTRPATH = f"{BASEPATH}/utrcalling"

PEAKS_PROPORTIONS_FILENAME = (
    f"{UTRPATH}/merged_bw12_above10cts_apa_named_percentages_annotated.h5ad"
)
RAW_UTR_COUNTS_FILENAME = f"{UTRPATH}/merged_raw_annotated.h5ad"

RESULTS_DIR = Path(f"{BASEPATH}/utrcalling")
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results dir: {RESULTS_DIR}")
print(f"Figure dir: {FIG_DIR}")

# Load data


In [ ]:
adata_raw = sc.read_h5ad(COUNTS_FILE)
adata_utr_raw = sc.read_h5ad(RAW_UTR_COUNTS_FILENAME)
adata_peaks_proportions = sc.read_h5ad(PEAKS_PROPORTIONS_FILENAME)

display(adata_raw)
display(adata_utr_raw)
display(adata_peaks_proportions)

## Check data


In [ ]:
display(adata_raw.obs.dtypes)
display(adata_utr_raw.obs.dtypes)
display(adata_peaks_proportions.obs.dtypes)

In [ ]:
celltypes = adata_peaks_proportions.obs.celltype.unique().tolist()
display(celltypes)

In [ ]:
lineages = {
    "Inhibitory neurons": [
        "Astrocytes",
        "vRG",
        "ccvRG",
        "INP",
        "LGE_IN",
        "CGE_LGE_IN",
        "CGE_IN",
    ],
    "Excitatory neurons": ["oRG", "RG", "ccRG", "IP", "L6_CThPN", "L56", "L4", "L23"],
}

In [ ]:
adata_peaks_proportions.var.gene_name

In [ ]:
adata_peaks_proportions.var.name

## Fixes


### Fix 1: Convert `batch_number` and `sample_id` to Categorical in UTR datasets

The HDF5-backed `.h5ad` format stores `pd.Categorical` columns natively (codes array +
categories array). They round-trip through `write_h5ad` / `read_h5ad` without losing their
dtype, so this change will persist in every downstream notebook that loads the saved
files - no extra steps are needed beyond saving below.


In [ ]:
for adata, label in [
    (adata_utr_raw, "adata_utr_raw"),
    (adata_peaks_proportions, "adata_peaks_proportions"),
]:
    for col in ["batch_number", "sample_id"]:
        if col in adata.obs.columns:
            adata.obs[col] = adata.obs[col].astype(str).astype("category")
            cats = adata.obs[col].cat.categories.tolist()
            print(f"  {label}.obs['{col}'] → category  {cats}")
        else:
            print(f"  WARNING: '{col}' not found in {label}.obs – skipping")

### Fix 2: Repair `gene_name` in `.var` of UTR datasets

Two problems to address:

1. Missing names (`None` / `"None"`) - fall back to the Ensembl ID stored in
   `var["name"]`.
2. `|`-prefixed names (e.g. `"|MAD1L1"`) - arise when a UTR site spans multiple gene
   bodies, making the annotation ambiguous. These features are removed from both UTR
   datasets rather than patched, as their counts cannot be unambiguously attributed to a
   single gene.


In [ ]:
def _fix_and_filter_var_gene_names(adata, ensembl_col: str = "name", label: str = ""):
    """
    Fix gene_name in .var in-place (None → Ensembl ID fallback), then return a
    filtered copy with '|'-prefixed entries removed (multi-gene-spanning UTR sites).

    Parameters
    ----------
    adata       : AnnData to process (.var must contain 'gene_name').
    ensembl_col : column in .var holding the Ensembl ID (fallback for None entries).
    label       : human-readable label for log messages.

    Returns
    -------
    AnnData - filtered copy (or the original if no pipe features were found).
    """
    if "gene_name" not in adata.var.columns:
        print(f"  {label}: no 'gene_name' column — skipping")
        return adata

    # Cast to plain object dtype so we can assign arbitrary new strings
    adata.var["gene_name"] = adata.var["gene_name"].astype(object)

    # ── 1. None / NaN → Ensembl ID ──────────────────────────────────────────
    missing_mask = adata.var["gene_name"].isna() | (
        adata.var["gene_name"].astype(str) == "None"
    )
    n_none = int(missing_mask.sum())
    if n_none:
        if ensembl_col not in adata.var.columns:
            print(
                f"  {label}: WARNING – '{ensembl_col}' not found; "
                f"{n_none} None entries remain"
            )
        else:
            adata.var.loc[missing_mask, "gene_name"] = (
                adata.var.loc[missing_mask, ensembl_col].astype(str).values
            )
            print(f"  {label}: {n_none} None gene_name entries → Ensembl ID fallback")
    else:
        print(f"  {label}: no None gene_name entries found")

    # ── 2. Remove '|'-prefixed features (multi-gene-spanning UTR sites) ──────
    gn = adata.var["gene_name"].astype(str)
    pipe_mask = gn.str.startswith("|")
    n_pipe = int(pipe_mask.sum())
    n_unique_pipe = int(gn[pipe_mask].nunique())
    if n_pipe:
        examples = gn[pipe_mask].unique()[:5].tolist()
        print(
            f"  {label}: removing {n_pipe} features across "
            f"{n_unique_pipe} unique '|'-prefixed gene_names "
            f"(multi-gene UTRs) — e.g. {examples}"
        )
        adata = adata[:, ~pipe_mask].copy()
    else:
        print(f"  {label}: no '|'-prefixed features found")

    return adata


print("Fixing gene_name in UTR datasets …\n")
adata_utr_raw = _fix_and_filter_var_gene_names(adata_utr_raw, label="adata_utr_raw")
adata_peaks_proportions = _fix_and_filter_var_gene_names(
    adata_peaks_proportions, label="adata_peaks_proportions"
)
print(f"\nFinal shapes after fixes:")
print(f"  adata_utr_raw:           {adata_utr_raw.shape}")
print(f"  adata_peaks_proportions: {adata_peaks_proportions.shape}")

## Add metadata


### Add `lineage` column to all 3 datasets

Map every cell's `celltype` to one of the two lineages defined in the `lineages` dict.
The resulting column is a `pd.Categorical` with categories ordered
Inhibitory neurons, then Excitatory neurons (matching the insertion order of the
`lineages` dict keys).


In [ ]:
# Build a flat celltype → lineage lookup from the lineages dict
celltype_to_lineage = {
    ct: lineage_name for lineage_name, cts in lineages.items() for ct in cts
}
# Preserve key insertion order: ["Inhibitory neurons", "Excitatory neurons"]
lineage_order = list(lineages.keys())

print("Assigning lineage column …\n")
all_ok = True
for adata, label in [
    (adata_raw, "adata_raw"),
    (adata_utr_raw, "adata_utr_raw"),
    (adata_peaks_proportions, "adata_peaks_proportions"),
]:
    adata.obs["lineage"] = adata.obs["celltype"].map(celltype_to_lineage)

    # ── Validate: every cell must receive a lineage ────────────────────────
    n_unmapped = int(adata.obs["lineage"].isna().sum())
    if n_unmapped:
        unmapped = (
            adata.obs.loc[adata.obs["lineage"].isna(), "celltype"].unique().tolist()
        )
        print(
            f"  WARNING {label}: {n_unmapped} cells have no lineage assignment. "
            f"Unknown celltypes: {unmapped}"
        )
        all_ok = False
    else:
        print(f"  {label}: all {adata.n_obs:,} cells assigned ✓")

    # ── Encode as Categorical (Inhibitory first, Excitatory second) ────────
    adata.obs["lineage"] = pd.Categorical(
        adata.obs["lineage"],
        categories=lineage_order,
        ordered=False,
    )

assert all_ok, (
    "One or more datasets contain cells that could not be assigned a lineage. "
    "Check the warnings above and update the lineages dict accordingly."
)
print("\nAll cells have a non-NA lineage value.")

### Order `celltype` categories in all 3 datasets

Categories are reordered to match the canonical lineage order defined in `lineages`:
Inhibitory neurons first (in the order listed there), then Excitatory neurons. This
ordering is stored as a `pd.Categorical` and persists through `write_h5ad` / `read_h5ad`.


In [ ]:
# Canonical celltype order: all inhibitory celltypes first, then all excitatory
celltype_order = [ct for cts in lineages.values() for ct in cts]

print("Reordering celltype categories …\n")
for adata, label in [
    (adata_raw, "adata_raw"),
    (adata_utr_raw, "adata_utr_raw"),
    (adata_peaks_proportions, "adata_peaks_proportions"),
]:
    current_cats = set(adata.obs["celltype"].cat.categories.tolist())
    ordered_cats_present = [ct for ct in celltype_order if ct in current_cats]
    unexpected = current_cats - set(ordered_cats_present)
    if unexpected:
        print(
            f"  WARNING {label}: the following celltypes are present in .obs "
            f"but not in the lineages dict and will be appended at the end: "
            f"{sorted(unexpected)}"
        )
        ordered_cats_present = ordered_cats_present + sorted(unexpected)

    adata.obs["celltype"] = pd.Categorical(
        adata.obs["celltype"],
        categories=ordered_cats_present,
        ordered=False,
    )
    print(f"  {label}: {len(ordered_cats_present)} categories ordered ✓")

print("\nFinal celltype category order (same for all 3 datasets):")
print(adata_raw.obs["celltype"].cat.categories.tolist())

## Save fixed datasets

Write all three AnnData objects to disk. The output filenames are derived from the
original filenames by inserting `_fix` immediately after `annotated` in the stem.


In [ ]:
def _make_fix_path(original_path: str) -> str:
    """Insert '_fix' after 'annotated' in the filename stem."""
    from pathlib import Path

    p = Path(original_path)
    new_stem = p.stem.replace("annotated", "annotated_fix", 1)
    return str(p.with_name(new_stem + p.suffix))


save_jobs = [
    (adata_raw, COUNTS_FILE),
    (adata_utr_raw, RAW_UTR_COUNTS_FILENAME),
    (adata_peaks_proportions, PEAKS_PROPORTIONS_FILENAME),
]

print("Saving …\n")
for adata, original_path in save_jobs:
    out_path = _make_fix_path(original_path)
    print(f"  {Path(original_path).name}")
    print(f"    → {Path(out_path).name}")
    adata.write_h5ad(out_path)
    print(f"    ✓ written  ({adata.shape[0]:,} cells × {adata.shape[1]:,} features)")

print("\nDone.")